# 03. Scoring models
Logistic regression on WoE (scorecard) and LightGBM, each with and without EXT_SOURCE_1/2/3.
Bins, WoE and feature selection are fitted on **train**; tuning and comparison use **valid**;
test is not loaded in this notebook (decision 23).

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

from src.viz import INK_MUTED, INK_SECONDARY, SOURCE_COLORS, apply_style, save_figure
from src.woe import MISSING, WoETransformer, fit_feature, select_features

MART_PATH = PROJECT_ROOT / "data" / "processed" / "data_mart.parquet"
FIGURES_DIR = PROJECT_ROOT / "reports" / "figures"
RANDOM_STATE = 42

apply_style()
pd.set_option("display.max_columns", 50, "display.width", 200, "display.max_rows", 100)

In [ ]:
mart = pd.read_parquet(MART_PATH)
train = mart[mart["sample"] == "train"].reset_index(drop=True)
valid = mart[mart["sample"] == "valid"].reset_index(drop=True)
del mart  # test stays untouched until the final evaluation

# CODE_GENDER is kept in the mart only for the fairness audit (decision 15)
NOT_FEATURES = ["sample", "SK_ID_CURR", "TARGET", "CODE_GENDER"]
FEATURES = [c for c in train.columns if c not in NOT_FEATURES]
EXT_FEATURES = ["EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3"]

assert len(train) == 184_506 and len(valid) == 61_502
print(f"{len(FEATURES)} candidate features; train {len(train):,}, valid {len(valid):,}")

## 3.2 Binning, WoE and Information Value (train)

In [ ]:
woe = WoETransformer(max_bins=20, min_share=0.05).fit(train[FEATURES], train["TARGET"])
iv = woe.iv_table()

# Price of monotonicity: IV of the same numeric feature with bins free to be U-shaped
iv_free = {
    f: fit_feature(train[f], train["TARGET"], monotonic=False).iv
    for f in iv.loc[iv["kind"] == "numeric", "feature"]
}
iv["iv_non_monotonic"] = iv["feature"].map(iv_free)
iv["iv_lost_by_monotonic"] = 1 - iv["iv"] / iv["iv_non_monotonic"]


def iv_strength(v: float) -> str:
    if v < 0.02:
        return "useless"
    if v < 0.1:
        return "weak"
    if v < 0.3:
        return "medium"
    if v < 0.5:
        return "strong"
    return "suspicious"


iv["strength"] = iv["iv"].map(iv_strength)
iv.round(4)

In [ ]:
# Features where forcing monotonic bins costs the most information
iv[iv["iv_lost_by_monotonic"] > 0.15].sort_values("iv_non_monotonic", ascending=False).round(4)

In [ ]:
# Example: full binning table of one feature
woe.binnings_["age_years"].table.round(4)

In [ ]:
PREFIX_TO_SOURCE = {"bureau_": "bureau", "prev_": "previous", "inst_": "installments",
                    "pos_": "pos", "cc_": "credit_card"}


def column_source(col: str) -> str:
    return next((s for p, s in PREFIX_TO_SOURCE.items() if col.startswith(p)), "application")


plot_df = iv.sort_values("iv")
fig, ax = plt.subplots(figsize=(9, 13))
bars = ax.barh(plot_df["feature"], plot_df["iv"],
               color=[SOURCE_COLORS[column_source(f)] for f in plot_df["feature"]], height=0.7)
for bar, v in zip(bars, plot_df["iv"]):
    ax.text(bar.get_width() + 0.004, bar.get_y() + bar.get_height() / 2, f"{v:.3f}",
            va="center", fontsize=7, color=INK_SECONDARY)
for threshold, label in [(0.02, "0.02 useless"), (0.1, "0.1 medium"), (0.3, "0.3 strong")]:
    ax.axvline(threshold, color=INK_MUTED, linestyle="--", linewidth=1)
    ax.text(threshold, 1.002, label, transform=ax.get_xaxis_transform(), ha="center",
            va="bottom", fontsize=8, color=INK_MUTED)
ax.grid(axis="y", visible=False)
ax.tick_params(axis="y", labelsize=8)
ax.set_xlabel("Information Value (train)")
ax.set_title("Information Value of candidate features", pad=18)
sources = [s for s in SOURCE_COLORS if s in {column_source(f) for f in plot_df["feature"]}]
ax.legend([plt.Rectangle((0, 0), 1, 1, color=SOURCE_COLORS[s]) for s in sources], sources,
          title="Source", loc="lower right")

save_figure(fig, "03_information_value", FIGURES_DIR)
plt.show()

In [ ]:
# Draft selection: IV >= 0.02, |corr of WoE| <= 0.7, greedy from the highest IV
woe_train = woe.transform(train[FEATURES])
iv_series = iv.set_index("feature")["iv"]

selection_ext = select_features(woe_train, iv_series, min_iv=0.02, max_corr=0.7)
no_ext = [f for f in FEATURES if f not in EXT_FEATURES]
selection_no_ext = select_features(woe_train[no_ext], iv_series[no_ext], min_iv=0.02, max_corr=0.7)

print("Selected with EXT_SOURCE:   ", selection_ext["selected"].sum())
print("Selected without EXT_SOURCE:", selection_no_ext["selected"].sum())
selection_ext.round(4)

## 3.3 Logistic regression on WoE (scorecard)
Final binning: monotonic WoE for every numeric feature, including `credit_term` (decision 25).
Free bins of `credit_term` add ~0.006-0.010 valid Gini, but its bad rate jumps between products
and terms (a saw, not a U-shape), which a scorecard cannot explain; LightGBM can use that signal.
Selection: IV >= 0.02, |corr| <= 0.7 (decision 26).

In [ ]:
from src.scorecard import fit_logreg_woe, gini, score, scorecard_table

woe = WoETransformer(max_bins=20, min_share=0.05)
woe.fit(train[FEATURES], train["TARGET"])
iv_final = woe.iv_table().set_index("feature")["iv"]

woe_train = woe.transform(train[FEATURES])
woe_valid = woe.transform(valid[FEATURES])  # bins learned on train, applied unchanged

NO_EXT = [f for f in FEATURES if f not in EXT_FEATURES]
selected = {
    "with_ext": select_features(woe_train, iv_final, min_iv=0.02, max_corr=0.7),
    "no_ext": select_features(woe_train[NO_EXT], iv_final[NO_EXT], min_iv=0.02, max_corr=0.7),
}
for version, sel in selected.items():
    print(version, "->", sel["selected"].sum(), "features")

In [ ]:
logreg = {}
rows = []
for version, sel in selected.items():
    candidates = sel.loc[sel["selected"], "feature"].tolist()
    model, features, dropped = fit_logreg_woe(woe_train, train["TARGET"], candidates, random_state=RANDOM_STATE)
    logreg[version] = (model, features)
    pd_train = model.predict_proba(woe_train[features])[:, 1]
    pd_valid = model.predict_proba(woe_valid[features])[:, 1]
    rows.append({
        "model": f"logreg_woe_{version}",
        "n_features": len(features),
        "dropped_wrong_sign": ", ".join(dropped) or "-",
        "gini_train": gini(train["TARGET"], pd_train),
        "gini_valid": gini(valid["TARGET"], pd_valid),
    })

results_logreg = pd.DataFrame(rows)
results_logreg["gini_drop"] = results_logreg["gini_train"] - results_logreg["gini_valid"]
results_logreg.round(4)

In [ ]:
# Coefficients of the model with external scores: all negative by construction
model, features = logreg["with_ext"]
coefs = pd.DataFrame({"feature": features, "coef": model.coef_[0], "iv": iv_final[features].values})
coefs.sort_values("coef").round(4)

In [ ]:
# Scorecard: points per bin; 600 points = odds 50:1 (good:bad), +20 points doubles the odds
card = scorecard_table(woe.binnings_, model, features)
card["points"] = card["points"].round(0).astype(int)
card[card["feature"].isin(["EXT_SOURCE_3", "age_years", "credit_term", "OCCUPATION_TYPE"])].round(4)

In [ ]:
from src.viz import BAD_COLOR, GOOD_COLOR

valid_score = score(woe_valid, model, features)
bins_hist = np.arange(np.floor(valid_score.min() / 10) * 10, valid_score.max() + 10, 10)

fig, ax = plt.subplots(figsize=(10, 4.8))
for label, mask, color in [("repaid (TARGET = 0)", valid["TARGET"] == 0, GOOD_COLOR),
                           ("defaulted (TARGET = 1)", valid["TARGET"] == 1, BAD_COLOR)]:
    ax.hist(valid_score[mask], bins=bins_hist, density=True, histtype="step", linewidth=2,
            color=color, label=f"{label}: median {valid_score[mask].median():.0f}")
ax.set_xlabel("Scorecard score (higher = safer)")
ax.set_ylabel("Density")
ax.set_title(f"Score distribution on valid, scorecard with external scores "
             f"(Gini {gini(valid['TARGET'], -valid_score):.3f})")
ax.legend(loc="upper right")

save_figure(fig, "03_score_distribution", FIGURES_DIR)
plt.show()

## 3.4 LightGBM
Raw features (no binning): trees find their own split points, handle NaN natively and
categorical features via the pandas `category` dtype. Early stopping on valid AUC chooses
the number of trees; valid Gini is therefore slightly optimistic, test gives the final estimate.

In [ ]:
import lightgbm as lgb

from src import modeling

# Text columns (dtype str in pandas 3, object in older versions)
CATEGORICAL = [c for c in FEATURES if not pd.api.types.is_numeric_dtype(train[c])]


CATEGORIES = modeling.category_levels(train, CATEGORICAL)  # learned on train only


def to_lgbm_frame(df: pd.DataFrame, features: list[str]) -> pd.DataFrame:
    """Categorical columns get the train categories, so codes match across samples."""
    return modeling.to_lgbm_frame(df, features, CATEGORIES)


LGBM_PARAMS = dict(
    n_estimators=3000,          # upper bound; early stopping picks the real number
    learning_rate=0.05,
    # Shallow trees (decision 28): same valid Gini as 31 leaves / 200 clients per leaf,
    # but half the train-valid gap; see the comparison of configurations below
    num_leaves=7,
    min_child_samples=1000,     # every leaf describes at least 1000 clients
    subsample=0.8, subsample_freq=1,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    random_state=RANDOM_STATE,
    verbose=-1,
)

lgbm, rows = {}, []
for version, features in [("with_ext", FEATURES), ("no_ext", NO_EXT)]:
    X_train, X_valid = to_lgbm_frame(train, features), to_lgbm_frame(valid, features)
    model = lgb.LGBMClassifier(**LGBM_PARAMS)
    model.fit(
        X_train, train["TARGET"],
        eval_X=X_valid, eval_y=valid["TARGET"], eval_metric="auc",
        callbacks=[lgb.early_stopping(100, verbose=False)],
    )
    lgbm[version] = (model, features)
    rows.append({
        "model": f"lightgbm_{version}",
        "n_features": len(features),
        "n_trees": model.best_iteration_,
        "gini_train": gini(train["TARGET"], model.predict_proba(X_train)[:, 1]),
        "gini_valid": gini(valid["TARGET"], model.predict_proba(X_valid)[:, 1]),
    })

results_lgbm = pd.DataFrame(rows)
results_lgbm["gini_drop"] = results_lgbm["gini_train"] - results_lgbm["gini_valid"]
results_lgbm.round(4)

In [ ]:
# Why shallow trees: valid Gini is flat across complexity, the train-valid gap is not
configs = [(31, 200, 0.05), (15, 500, 0.05), (7, 1000, 0.05), (15, 1000, 0.02)]
X_train, X_valid = to_lgbm_frame(train, FEATURES), to_lgbm_frame(valid, FEATURES)
rows = []
for leaves, min_child, lr in configs:
    m = lgb.LGBMClassifier(**{**LGBM_PARAMS, "num_leaves": leaves, "min_child_samples": min_child,
                              "learning_rate": lr, "n_estimators": 5000})
    m.fit(X_train, train["TARGET"], eval_X=X_valid, eval_y=valid["TARGET"], eval_metric="auc",
          callbacks=[lgb.early_stopping(100, verbose=False)])
    g_train = gini(train["TARGET"], m.predict_proba(X_train)[:, 1])
    g_valid = gini(valid["TARGET"], m.predict_proba(X_valid)[:, 1])
    rows.append({"num_leaves": leaves, "min_child_samples": min_child, "learning_rate": lr,
                 "n_trees": m.best_iteration_, "gini_train": g_train, "gini_valid": g_valid,
                 "gini_drop": g_train - g_valid})
pd.DataFrame(rows).round(4)

In [ ]:
# SHAP values (TreeSHAP built into LightGBM) on a fixed sample of valid clients.
# Importance = mean |SHAP|: average shift of the log-odds of default caused by the feature.
model, features = lgbm["with_ext"]
sample = valid.sample(20_000, random_state=RANDOM_STATE)
contrib = model.predict(to_lgbm_frame(sample, features), pred_contrib=True)  # last column = base value
shap_values = pd.DataFrame(contrib[:, :-1], columns=features, index=sample.index)

importance = (
    shap_values.abs().mean().sort_values(ascending=False)
    .rename("mean_abs_shap").to_frame()
    .join(iv_final.rename("iv"))
)
importance.head(20).round(4)

In [ ]:
top = importance.head(20).iloc[::-1]
fig, ax = plt.subplots(figsize=(9, 7))
bars = ax.barh(top.index, top["mean_abs_shap"],
               color=[SOURCE_COLORS[column_source(f)] for f in top.index], height=0.7)
for bar, v in zip(bars, top["mean_abs_shap"]):
    ax.text(bar.get_width() + 0.003, bar.get_y() + bar.get_height() / 2, f"{v:.3f}",
            va="center", fontsize=8, color=INK_SECONDARY)
ax.grid(axis="y", visible=False)
ax.set_xlabel("Mean |SHAP value| (log-odds of default), valid sample of 20 000")
ax.set_title("LightGBM with external scores: top 20 features by SHAP importance")
sources = [s for s in SOURCE_COLORS if s in {column_source(f) for f in top.index}]
ax.legend([plt.Rectangle((0, 0), 1, 1, color=SOURCE_COLORS[s]) for s in sources], sources,
          title="Source", loc="lower right")

save_figure(fig, "03_shap_importance", FIGURES_DIR)
plt.show()

## 3.5 Model comparison and calibration (valid)
Gini with 95% bootstrap intervals, ROC curves, calibration of PD by decile.
PD values feed the economics of stage 4 (expected loss = PD x LGD x EAD), so ranking alone is not enough.

In [ ]:
MODEL_LABELS = {
    "scorecard_with_ext": "Scorecard, with EXT_SOURCE",
    "scorecard_no_ext": "Scorecard, no EXT_SOURCE",
    "lightgbm_with_ext": "LightGBM, with EXT_SOURCE",
    "lightgbm_no_ext": "LightGBM, no EXT_SOURCE",
}

pd_train, pd_valid = {}, {}
for version, (m, f) in logreg.items():
    pd_train[f"scorecard_{version}"] = m.predict_proba(woe_train[f])[:, 1]
    pd_valid[f"scorecard_{version}"] = m.predict_proba(woe_valid[f])[:, 1]
for version, (m, f) in lgbm.items():
    pd_train[f"lightgbm_{version}"] = m.predict_proba(to_lgbm_frame(train, f))[:, 1]
    pd_valid[f"lightgbm_{version}"] = m.predict_proba(to_lgbm_frame(valid, f))[:, 1]

n_features = {f"scorecard_{v}": len(f) for v, (_, f) in logreg.items()} | \
             {f"lightgbm_{v}": len(f) for v, (_, f) in lgbm.items()}
comparison = pd.DataFrame({
    "n_features": n_features,
    "gini_train": {k: gini(train["TARGET"], v) for k, v in pd_train.items()},
    "gini_valid": {k: gini(valid["TARGET"], v) for k, v in pd_valid.items()},
}).loc[list(MODEL_LABELS)]
comparison["gini_drop"] = comparison["gini_train"] - comparison["gini_valid"]
comparison.round(4)

In [ ]:
# Bootstrap on valid: resample clients with replacement, recompute Gini of every model on the
# same resample, so the differences between models are paired
N_BOOT = 500
rng = np.random.default_rng(RANDOM_STATE)
y_valid = valid["TARGET"].to_numpy()
boot_rows = []
for _ in range(N_BOOT):
    idx = rng.integers(0, len(y_valid), len(y_valid))
    boot_rows.append({k: gini(y_valid[idx], p[idx]) for k, p in pd_valid.items()})
boot = pd.DataFrame(boot_rows)

DIFFS = {
    "LightGBM - scorecard (with EXT)": ("lightgbm_with_ext", "scorecard_with_ext"),
    "LightGBM - scorecard (no EXT)": ("lightgbm_no_ext", "scorecard_no_ext"),
    "with EXT - no EXT (scorecard)": ("scorecard_with_ext", "scorecard_no_ext"),
    "with EXT - no EXT (LightGBM)": ("lightgbm_with_ext", "lightgbm_no_ext"),
}


def ci_row(name: str, point: float, samples: pd.Series) -> dict:
    return {"name": name, "gini": point, "ci_low": samples.quantile(0.025), "ci_high": samples.quantile(0.975)}


gini_ci = pd.DataFrame([ci_row(k, comparison.loc[k, "gini_valid"], boot[k]) for k in MODEL_LABELS])
diff_ci = pd.DataFrame([
    ci_row(name, comparison.loc[a, "gini_valid"] - comparison.loc[b, "gini_valid"], boot[a] - boot[b])
    for name, (a, b) in DIFFS.items()
])
display(gini_ci.round(4))
diff_ci.round(4)

In [ ]:
from src.viz import SERIES

# Gini on valid with 95% bootstrap CI: groups = model type, colour = with / without external scores
VERSION_COLORS = {"with_ext": SERIES[0], "no_ext": SERIES[1]}
VERSION_LABELS = {"with_ext": "with EXT_SOURCE", "no_ext": "without EXT_SOURCE"}
ci = gini_ci.set_index("name")

fig, ax = plt.subplots(figsize=(8, 4.8))
width = 0.36
for j, version in enumerate(["with_ext", "no_ext"]):
    keys = [f"scorecard_{version}", f"lightgbm_{version}"]
    x = np.arange(2) + (j - 0.5) * width
    vals = ci.loc[keys, "gini"]
    ax.bar(x, vals, width=width * 0.92, color=VERSION_COLORS[version], label=VERSION_LABELS[version])
    ax.errorbar(x, vals, yerr=[vals - ci.loc[keys, "ci_low"], ci.loc[keys, "ci_high"] - vals],
                fmt="none", ecolor=INK_SECONDARY, elinewidth=1, capsize=3)
    for xi, v, hi in zip(x, vals, ci.loc[keys, "ci_high"]):
        ax.text(xi, hi + 0.008, f"{v:.3f}", ha="center", fontsize=9, color=INK_SECONDARY)
ax.set_xticks(np.arange(2), ["Scorecard\n(logistic regression on WoE)", "LightGBM"])
ax.set_ylim(0, 0.65)
ax.set_ylabel("Gini on valid")
ax.grid(axis="x", visible=False)
ax.set_title("Model comparison on valid; whiskers: 95% bootstrap CI (500 resamples)")
ax.legend(loc="upper left")

save_figure(fig, "03_model_comparison", FIGURES_DIR)
plt.show()

In [ ]:
from sklearn.metrics import roc_curve

# Colour = external scores or not, line style = model type
STYLE = {"scorecard": "--", "lightgbm": "-"}
fig, ax = plt.subplots(figsize=(6.5, 6))
for key, label in MODEL_LABELS.items():
    model_type, version = key.split("_", 1)
    fpr, tpr, _ = roc_curve(y_valid, pd_valid[key])
    ax.plot(fpr, tpr, color=VERSION_COLORS[version], linestyle=STYLE[model_type],
            label=f"{label}: Gini {comparison.loc[key, 'gini_valid']:.3f}")
ax.plot([0, 1], [0, 1], color=INK_MUTED, linewidth=1, linestyle=":", label="random model: Gini 0")
ax.set_xlabel("False positive rate (good clients flagged as risky)")
ax.set_ylabel("True positive rate (defaults caught)")
ax.set_title("ROC curves on valid")
ax.legend(loc="lower right", fontsize=8)
ax.set_aspect("equal")

save_figure(fig, "03_roc_curves", FIGURES_DIR)
plt.show()

In [ ]:
from sklearn.metrics import brier_score_loss

# Calibration: within each decile of predicted PD, mean PD should equal the observed bad rate
calib_rows, deciles = [], {}
for key in MODEL_LABELS:
    d = pd.DataFrame({"pd": pd_valid[key], "y": y_valid})
    d["decile"] = pd.qcut(d["pd"], 10, labels=False, duplicates="drop")
    t = d.groupby("decile").agg(mean_pd=("pd", "mean"), bad_rate=("y", "mean"), n=("y", "size"))
    deciles[key] = t
    calib_rows.append({
        "model": key,
        "mean_pd": d["pd"].mean(),
        "bad_rate": d["y"].mean(),
        "mean_pd_to_bad_rate": d["pd"].mean() / d["y"].mean(),
        "max_decile_gap_pp": (t["mean_pd"] - t["bad_rate"]).abs().max() * 100,
        "brier": brier_score_loss(d["y"], d["pd"]),
        "brier_base_rate_model": brier_score_loss(d["y"], np.full(len(d), d["y"].mean())),
    })
calibration = pd.DataFrame(calib_rows).set_index("model")
calibration.round(4)

In [ ]:
from matplotlib.ticker import PercentFormatter

fig, axes = plt.subplots(2, 2, figsize=(10, 9), sharex=True, sharey=True)
for ax, (key, label) in zip(axes.flat, MODEL_LABELS.items()):
    t = deciles[key]
    lim = max(t["mean_pd"].max(), t["bad_rate"].max()) * 1.1
    ax.plot([0, 0.3], [0, 0.3], color=INK_MUTED, linewidth=1, linestyle=":")
    ax.plot(t["mean_pd"], t["bad_rate"], color=VERSION_COLORS[key.split("_", 1)[1]], marker="o")
    ax.set_title(f"{label}\nBrier {calibration.loc[key, 'brier']:.4f}", fontsize=10)
    ax.xaxis.set_major_formatter(PercentFormatter(1.0, decimals=0))
    ax.yaxis.set_major_formatter(PercentFormatter(1.0, decimals=0))
for ax in axes[1]:
    ax.set_xlabel("Mean predicted PD in decile")
for ax in axes[:, 0]:
    ax.set_ylabel("Observed bad rate in decile")
axes[0, 0].set_xlim(0, 0.3)
axes[0, 0].set_ylim(0, 0.3)
fig.suptitle("Calibration on valid: deciles of predicted PD; dotted line = perfect calibration",
             x=0.01, ha="left", fontsize=12, fontweight="bold")
fig.tight_layout()

save_figure(fig, "03_calibration", FIGURES_DIR)
plt.show()

In [ ]:
# Save fitted models and valid predictions for stage 4 (data/ is git-ignored)
MODELS_DIR = PROJECT_ROOT / "data" / "processed" / "models"
for version, (m, f) in logreg.items():
    modeling.save_bundle(MODELS_DIR / f"scorecard_{version}.joblib", woe=woe, model=m, features=f)
for version, (m, f) in lgbm.items():
    modeling.save_bundle(MODELS_DIR / f"lightgbm_{version}.joblib", model=m, features=f, categories=CATEGORIES)

# Reload check: the saved bundles reproduce the PD on valid
for key in MODEL_LABELS:
    b = modeling.load_bundle(MODELS_DIR / f"{key}.joblib")
    if key.startswith("scorecard"):
        reloaded = b["model"].predict_proba(b["woe"].transform(valid[FEATURES])[b["features"]])[:, 1]
    else:
        X = modeling.to_lgbm_frame(valid, b["features"], b["categories"])
        reloaded = b["model"].predict_proba(X)[:, 1]
    assert np.allclose(reloaded, pd_valid[key]), key

model_wext, features_wext = logreg["with_ext"]
valid_predictions = valid[["SK_ID_CURR", "TARGET", "AMT_CREDIT", "AMT_ANNUITY", "AMT_INCOME_TOTAL",
                           "CODE_GENDER", "age_years"]].copy()
for key in MODEL_LABELS:
    valid_predictions[f"pd_{key}"] = pd_valid[key]
valid_predictions["score_scorecard_with_ext"] = score(woe_valid, model_wext, features_wext).to_numpy()

assert len(valid_predictions) == 61_502
assert valid_predictions.filter(like="pd_").notna().all().all()
valid_predictions.to_parquet(PROJECT_ROOT / "data" / "processed" / "valid_predictions.parquet", index=False)
print("Saved", len(valid_predictions), "rows and", len(MODEL_LABELS), "model bundles")

In [ ]:
# Feature summary of the final scorecard for docs/FEATURES.md
card_wext = scorecard_table(woe.binnings_, model_wext, features_wext)
no_ext_features = set(logreg["no_ext"][1])
feature_summary = (
    card_wext.groupby("feature", sort=False)
    .agg(points_min=("points", "min"), points_max=("points", "max"), n_bins=("bin", "size"))
    .assign(points_range=lambda t: t["points_max"] - t["points_min"])
    .join(pd.Series(model_wext.coef_[0], index=features_wext, name="coef"))
    .join(iv_final.rename("iv"))
)
feature_summary["source"] = [column_source(f) for f in feature_summary.index]
feature_summary["in_no_ext_model"] = [f in no_ext_features for f in feature_summary.index]
feature_summary.sort_values("points_range", ascending=False).round(3)